# R Master v5 · 衣物控制结构扫描（一键版）

v4 已确认：衣物并不是可以按材质槽或连通组件直接剥掉的独立壳。

v5 只扫描控制层，不修改模型：
- Modifier 的 vertex group / target；
- Mona_Main 的 Shape Keys 及每个 Shape Key 影响区域；
- Vertex Groups（重点找 Pant / Sleeve / Shoe / Belt / Glove / Cloth 等）；
- 相关骨骼；
- 驱动器 data_path；
- 所有 Mesh 对象名称与绑定关系。

它直接读取：
`MyDrive/R_Master/v2/latest/R_Master_Align_v2_PREVIEW.blend`

跑完只下载一个很小的：
`R_Master_v5_Report.zip`

目的：锁定“衣服到底由什么控制”，然后 v6 才做安全剥离。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, subprocess, json, zipfile, os

print("R Master v5 · 衣物控制结构扫描")
drive.mount("/content/drive")

ROOT=Path("/content/drive/MyDrive/R_Master")
SRC=ROOT/"v2"/"latest"/"R_Master_Align_v2_PREVIEW.blend"
CACHE=ROOT/"cache"
OUT=ROOT/"v5_controls"/"latest"
CACHE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)

if not SRC.exists() or SRC.stat().st_size < 50*1024*1024:
    raise RuntimeError("没找到 v2 预览文件。请把这屏截图给二蛋。")

BLENDER_VERSION="4.4.3"
BLENDER_URL="https://download.blender.org/release/Blender4.4/blender-4.4.3-linux-x64.tar.xz"
LOCAL=Path("/content/r_master_v5")
LOCAL.mkdir(parents=True,exist_ok=True)
ARCHIVE=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
BDIR=LOCAL/f"blender-{BLENDER_VERSION}-linux-x64"
DRIVE_ARCHIVE=CACHE/ARCHIVE.name

if shutil.which("xvfb-run") is None:
    subprocess.run(["apt-get","update","-qq"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)
    subprocess.run(["apt-get","install","-y","-qq","xvfb","libgl1","libx11-6","libxi6","libxrender1","libxfixes3","libxkbcommon0","libsm6"],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.STDOUT)

if DRIVE_ARCHIVE.exists() and DRIVE_ARCHIVE.stat().st_size>100*1024*1024:
    shutil.copy2(DRIVE_ARCHIVE,ARCHIVE)
    print("✓ 复用 Blender Drive 缓存")
else:
    subprocess.run(["wget","-q","--show-progress","-O",str(ARCHIVE),BLENDER_URL],check=True)
    shutil.copy2(ARCHIVE,DRIVE_ARCHIVE)

if not (BDIR/"blender").exists():
    if BDIR.exists(): shutil.rmtree(BDIR)
    subprocess.run(["tar","-xf",str(ARCHIVE),"-C",str(LOCAL)],check=True)
BLENDER=BDIR/"blender"
print("✓ Blender 就绪")

SCRIPT=LOCAL/"R_Master_v5_Scan.py"
SCRIPT.write_text(r"""
import bpy, os, sys, json, math, csv
from collections import defaultdict
from mathutils import Vector

argv=sys.argv[sys.argv.index("--")+1:] if "--" in sys.argv else []
out=None
for i,a in enumerate(argv):
    if a=="--out" and i+1<len(argv): out=argv[i+1]
if not out: raise RuntimeError("missing --out")
os.makedirs(out,exist_ok=True)

KEYWORDS=[
    "cloth","clothes","shirt","top","tank","pant","pants","short","shorts",
    "sleeve","shoe","boot","sock","belt","glove","wrist","bra","skirt",
    "dress","jacket","coat","hood","collar","under","wear"
]
def hit(name):
    n=(name or "").lower()
    return [k for k in KEYWORDS if k in n]

body=bpy.data.objects.get("R2_Mona_Main") or bpy.data.objects.get("Mona_Main")
if not body or body.type!="MESH":
    raise RuntimeError("找不到 R2_Mona_Main / Mona_Main")
mesh=body.data
mw=body.matrix_world

def bbox_from_vids(vids):
    vids=list(vids)
    if not vids: return None
    pts=[mw @ mesh.vertices[i].co for i in vids]
    mn=Vector((min(p.x for p in pts),min(p.y for p in pts),min(p.z for p in pts)))
    mx=Vector((max(p.x for p in pts),max(p.y for p in pts),max(p.z for p in pts)))
    c=(mn+mx)*0.5; e=mx-mn
    return {"min":list(map(float,mn)),"max":list(map(float,mx)),"center":list(map(float,c)),"extent":list(map(float,e))}

# modifiers
mods=[]
for m in body.modifiers:
    row={
        "name":m.name,"type":m.type,
        "show_viewport":bool(m.show_viewport),"show_render":bool(m.show_render),
        "keyword_hits":hit(m.name)
    }
    for attr in ["vertex_group","target","object","levels","render_levels","strength"]:
        if hasattr(m,attr):
            v=getattr(m,attr)
            if hasattr(v,"name"): v=v.name
            try: json.dumps(v)
            except: v=str(v)
            row[attr]=v
    mods.append(row)

# vertex groups
vgroups=[]
for vg in body.vertex_groups:
    vids=[]; total=0.0; maxw=0.0
    for v in mesh.vertices:
        try:
            w=vg.weight(v.index)
        except RuntimeError:
            continue
        if w>0:
            vids.append(v.index); total+=float(w); maxw=max(maxw,float(w))
    vgroups.append({
        "index":vg.index,"name":vg.name,"keyword_hits":hit(vg.name),
        "vertex_count":len(vids),"weight_sum":total,"max_weight":maxw,
        "bbox":bbox_from_vids(vids)
    })

# shape keys + approximate affected bbox from Basis delta
shapes=[]
if mesh.shape_keys and mesh.shape_keys.key_blocks:
    kb=mesh.shape_keys.key_blocks
    basis=kb[0]
    for key in kb:
        affected=[]
        max_delta=0.0
        sum_delta=0.0
        if key.name!=basis.name and len(key.data)==len(basis.data):
            for i,(a,b) in enumerate(zip(key.data,basis.data)):
                d=(a.co-b.co).length
                if d>1e-5:
                    affected.append(i)
                    max_delta=max(max_delta,float(d))
                    sum_delta+=float(d)
        shapes.append({
            "name":key.name,
            "value":float(key.value),
            "slider_min":float(key.slider_min),
            "slider_max":float(key.slider_max),
            "relative_key":key.relative_key.name if getattr(key,"relative_key",None) else None,
            "keyword_hits":hit(key.name),
            "affected_vertex_count":len(affected),
            "max_delta":max_delta,
            "sum_delta":sum_delta,
            "bbox":bbox_from_vids(affected)
        })

# bones / pose bones
arm=bpy.data.objects.get("R_Master_Align_v2_PREVIEW") or bpy.data.objects.get("Mona_Armature")
bones=[]
if arm and arm.type=="ARMATURE":
    for b in arm.data.bones:
        h=hit(b.name)
        if h:
            bones.append({
                "name":b.name,"keyword_hits":h,
                "head_local":list(map(float,b.head_local)),
                "tail_local":list(map(float,b.tail_local)),
                "parent":b.parent.name if b.parent else None
            })

# drivers across objects and data
drivers=[]
def collect_drivers(owner_label, owner):
    ad=getattr(owner,"animation_data",None)
    if not ad or not ad.drivers: return
    for f in ad.drivers:
        path=f.data_path
        h=hit(path)
        drivers.append({
            "owner":owner_label,
            "data_path":path,
            "array_index":int(f.array_index),
            "keyword_hits":h,
            "expression":getattr(f.driver,"expression",None)
        })
collect_drivers("body_object",body)
collect_drivers("body_data",mesh)
if mesh.shape_keys: collect_drivers("body_shape_keys",mesh.shape_keys)
if arm:
    collect_drivers("armature_object",arm)
    collect_drivers("armature_data",arm.data)

# all mesh objects overview
mesh_objects=[]
for obj in bpy.data.objects:
    if obj.type!="MESH": continue
    arm_mods=[m.object.name for m in obj.modifiers if m.type=="ARMATURE" and m.object]
    mesh_objects.append({
        "name":obj.name,
        "data_name":obj.data.name,
        "vertex_count":len(obj.data.vertices),
        "face_count":len(obj.data.polygons),
        "parent":obj.parent.name if obj.parent else None,
        "armature_modifiers":arm_mods,
        "keyword_hits":hit(obj.name+" "+obj.data.name),
        "hidden_viewport":bool(obj.hide_viewport),
        "hidden_render":bool(obj.hide_render)
    })

# custom props likely toggles
custom_props=[]
for label,obj in [("body",body),("armature",arm)]:
    if not obj: continue
    for k in obj.keys():
        if k.startswith("_"): continue
        try: v=obj[k]
        except: continue
        custom_props.append({"owner":label,"name":k,"value":str(v),"keyword_hits":hit(k)})

report={
    "ok":True,
    "stage":"R_Master_v5_ClothingControlScan",
    "body_object":body.name,
    "mesh_data":mesh.name,
    "counts":{
        "modifiers":len(mods),
        "vertex_groups":len(vgroups),
        "shape_keys":len(shapes),
        "keyword_bones":len(bones),
        "drivers":len(drivers),
        "mesh_objects":len(mesh_objects),
        "custom_props":len(custom_props)
    },
    "modifiers":mods,
    "vertex_groups":vgroups,
    "shape_keys":shapes,
    "keyword_bones":bones,
    "drivers":drivers,
    "mesh_objects":mesh_objects,
    "custom_props":custom_props
}
with open(os.path.join(out,"R_Master_v5_controls.json"),"w",encoding="utf-8") as f:
    json.dump(report,f,ensure_ascii=False,indent=2)

# focused CSVs
with open(os.path.join(out,"R_Master_v5_vertex_groups.csv"),"w",newline="",encoding="utf-8-sig") as f:
    w=csv.writer(f); w.writerow(["name","keyword_hits","vertex_count","weight_sum","max_weight","center","extent"])
    for x in vgroups:
        b=x["bbox"] or {}
        w.writerow([x["name"],"|".join(x["keyword_hits"]),x["vertex_count"],x["weight_sum"],x["max_weight"],b.get("center"),b.get("extent")])
with open(os.path.join(out,"R_Master_v5_shape_keys.csv"),"w",newline="",encoding="utf-8-sig") as f:
    w=csv.writer(f); w.writerow(["name","keyword_hits","value","slider_min","slider_max","affected_vertex_count","max_delta","sum_delta","center","extent"])
    for x in shapes:
        b=x["bbox"] or {}
        w.writerow([x["name"],"|".join(x["keyword_hits"]),x["value"],x["slider_min"],x["slider_max"],x["affected_vertex_count"],x["max_delta"],x["sum_delta"],b.get("center"),b.get("extent")])

print("[R Master v5] SCAN_OK")
print("[R Master v5] counts:",report["counts"])
print("[R Master v5] keyword modifiers:",[x for x in mods if x["keyword_hits"] or x.get("vertex_group")])
print("[R Master v5] keyword vertex groups:",[(x["name"],x["vertex_count"],x["keyword_hits"]) for x in vgroups if x["keyword_hits"]][:80])
print("[R Master v5] keyword shape keys:",[(x["name"],x["affected_vertex_count"],x["keyword_hits"]) for x in shapes if x["keyword_hits"]][:80])
print("[R Master v5] keyword bones:",[x["name"] for x in bones][:120])
print("[R Master v5] keyword drivers:",[(x["owner"],x["data_path"]) for x in drivers if x["keyword_hits"]][:80])
print("[R Master v5] keyword mesh objects:",[(x["name"],x["vertex_count"],x["face_count"]) for x in mesh_objects if x["keyword_hits"]][:80])
""",encoding="utf-8")

for p in OUT.iterdir():
    if p.is_file(): p.unlink()

print("③ 扫描衣物控制层…")
LOG=OUT/"R_Master_v5_blender.log"
cmd=["xvfb-run","-a",str(BLENDER),"--background",str(SRC),"--python",str(SCRIPT),"--","--out",str(OUT)]
with LOG.open("w",encoding="utf-8") as log:
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout:
        log.write(line)
        if "R Master v5" in line or "Traceback" in line or "Error" in line:
            print(line.rstrip())
    rc=p.wait()
if rc!=0:
    print(LOG.read_text(encoding="utf-8",errors="replace")[-10000:])
    raise RuntimeError(f"v5 扫描失败，退出码 {rc}。截图给二蛋即可。")

required=[
    OUT/"R_Master_v5_controls.json",
    OUT/"R_Master_v5_vertex_groups.csv",
    OUT/"R_Master_v5_shape_keys.csv",
    LOG
]
missing=[p.name for p in required if not p.exists()]
if missing: raise RuntimeError("v5 缺少输出："+", ".join(missing))

report=json.loads((OUT/"R_Master_v5_controls.json").read_text(encoding="utf-8"))
print("\n✓ R Master v5 SCAN_OK")
print("  统计：",report["counts"])
print("  衣物关键词骨骼：",[x["name"] for x in report["keyword_bones"]])
print("  衣物关键词 Vertex Groups：",[x["name"] for x in report["vertex_groups"] if x["keyword_hits"]])
print("  衣物关键词 Shape Keys：",[x["name"] for x in report["shape_keys"] if x["keyword_hits"]])

ZIP=OUT/"R_Master_v5_Report.zip"
with zipfile.ZipFile(ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in required: z.write(p,arcname=p.name)
print(f"\n✓ 报告包：{ZIP.stat().st_size/1024:.1f} KiB")
files.download(str(ZIP))

